# PCOSTraj Tutorial: Running the Full Pipeline

This notebook demonstrates the "happy path" for the PCOSTraj project:
- Load example gene expression data
- Train a classification model
- Extract important genes
- Save results

No validation or edge case handling is included.

In [ ]:
# Imports
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

In [ ]:
# Generate Example Dataset
np.random.seed(42)

n_samples = 100
n_genes = 300

# Generate synthetic gene expression data
X = np.random.normal(loc=5, scale=2, size=(n_samples, n_genes))

# Binary labels (0 = control, 1 = PCOS)
y = np.random.randint(0, 2, size=n_samples)

# Create DataFrame
columns = [f"Gene_{i+1}" for i in range(n_genes)]
df = pd.DataFrame(X, columns=columns)

df["label"] = y
df.insert(0, "sample_id", [f"S{i+1}" for i in range(n_samples)])

# Save dataset
df.to_csv("../data/example_dataset.csv", index=False)

df.head()

In [ ]:
# Load dataset
df = pd.read_csv("../data/example_dataset.csv")

print("Dataset shape:", df.shape)
df.head()

In [ ]:
# Prepare Features and Labels
X = df.drop(columns=["sample_id", "label"])
y = df["label"]

print("Feature matrix shape:", X.shape)
print("Labels shape:", y.shape)

In [ ]:
# Test/Train split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
# Train the Model
model = RandomForestClassifier(random_state=42)

model.fit(X_train, y_train)

print("Model training complete.")

In [ ]:
# Extract Feature Importance
importances = model.feature_importances_

feature_importance_df = pd.DataFrame({
    "Gene": X.columns,
    "Importance": importances
})

# Sort by importance
feature_importance_df = feature_importance_df.sort_values(
    by="Importance", ascending=False
)

feature_importance_df.head()

In [ ]:
# Get top 20 genes
top_genes = feature_importance_df.head(20)

top_genes

In [ ]:
# Create results folder if not exists
import os
os.makedirs("../results", exist_ok=True)

top_genes.to_csv("../results/top_genes.csv", index=False)

print("Top genes saved to ../results/top_genes.csv")

In [ ]:
# Plot Feature Importance
import matplotlib.pyplot as plt

top_genes.plot(
    kind="barh",
    x="Gene",
    y="Importance"
)

plt.gca().invert_yaxis()
plt.title("Top 20 Important Genes")
plt.xlabel("Importance Score")
plt.ylabel("Gene")

plt.show()